# Error Case 14 — 500: Duplicate Grant (Polaris Bug)

**Error:** `500 Internal Server Error` (not 409 as expected)
**Root cause:** Same privilege granted twice → DB unique constraint violation
               → not caught at service layer → RuntimeException → 500

## Actual Polaris 1.3.0 behavior

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [2]:
# Cell 2: Setup
r1 = create_catalog()
assert r1.status_code == 201, r1.text
r2 = create_catalog_role()
assert r2.status_code == 201

# First grant — succeeds
r3 = grant_privilege(privilege='CATALOG_MANAGE_CONTENT')
print_response(r3, 'First grant (should be 201)')
assert r3.status_code == 201
print('\n✅ First grant succeeded')


First grant (should be 201):
  Status:     201
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000169
  Body: 

✅ First grant succeeded


In [4]:
# Cell 3: Trigger 500 — grant same privilege again (Polaris bug)
r = grant_privilege(privilege='CATALOG_MANAGE_CONTENT')
print_response(r, '500 Duplicate grant (Polaris bug)')
assert r.status_code == 500           # ← 500 not 409
print('\n✅ 500 confirmed — duplicate grant causes RuntimeException')
print('\n⚠️  This is a known Polaris 1.3.0 bug')
print('   Duplicate grant → DB unique constraint violation')
print('   → not caught at service layer → 500 RuntimeException')
print('   Expected behavior: 200/201 (idempotent) or 409 (conflict)')


500 Duplicate grant (Polaris bug):
  Status:     500
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000173
  Body: {
  "error": {
    "message": "Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint \"grant_records_pkey\"\n  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)=(POLARIS, 0, 8994087783429165206, 8994087783429165206, 8019902197599819595, 32) already exists.' (error code 0, sql-state '23505'), after 1 attempts and 5000 milliseconds",
    "type": "RuntimeException",
    "code": 500
  }
}

✅ 500 confirmed — duplicate grant causes RuntimeException

⚠️  This is a known Polaris 1.3.0 bug
   Duplicate grant → DB unique constraint violation
   → not caught at service layer → 500 RuntimeException
   Expected behavior: 200/201 (idempotent) or 409 (conflict)


In [5]:
# Cell 4: Find error logs
wait_for_logs(5)
print('Searching for duplicate grant errors (last 1 min)...')

# 500 with duplicate key in message
hits = search_logs(message='duplicate key', minutes_ago=1)
if not hits:
    hits = search_logs(message='grant_records_pkey', minutes_ago=1)
if not hits:
    hits = search_500_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for duplicate grant errors (last 1 min)...
Found 2 log entries:

ℹ️  [2026-06-02T06:08:37.699Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000173
   realmId   : POLARIS
   message   : Handling runtimeException Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_i

ℹ️  [2026-06-02T06:08:37.699Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000173
   realmId   : POLARIS
   message   : Handling runtimeException Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_i



In [6]:
# Cell 5: Fix — check grant existence before adding
def safe_grant_privilege(catalog=TEST_CATALOG, cr=TEST_CR,
                         privilege='CATALOG_MANAGE_CONTENT', token=None):
    """
    Idempotent grant — checks existence before adding.
    Workaround for Polaris 1.3.0 duplicate grant bug.
    """
    # Check current grants
    r_check = requests.get(
        f'{BASE_MGMT}/catalogs/{catalog}/catalog-roles/{cr}/grants',
        headers=h(token)
    )
    if r_check.status_code == 200:
        existing = r_check.json().get('grants', [])
        for g in existing:
            if g.get('privilege') == privilege:
                print(f'Grant already exists: {privilege} — skipping')
                return r_check  # already granted

    # Grant not found — add it
    return grant_privilege(catalog, cr, privilege, token)

# Test idempotent grant
r_fix1 = safe_grant_privilege()   # already exists
r_fix2 = safe_grant_privilege()   # already exists again
print('\n✅ Idempotent grant works — no 409')

cleanup()
print('\n=== Summary ===')
print('Error:    409 Conflict')
print('Cause:    Polaris 1.3.0 bug — grant not idempotent')
print('Detect:   409 on PUT /grants with duplicate key error')
print('Fix:      GET /grants first, only PUT if not exists')
print('Pattern:  Use safe_grant_privilege() in all ETL setup scripts')
print('Status:   Known bug — may be fixed in future Polaris versions')

Grant already exists: CATALOG_MANAGE_CONTENT — skipping
Grant already exists: CATALOG_MANAGE_CONTENT — skipping

✅ Idempotent grant works — no 409
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    409 Conflict
Cause:    Polaris 1.3.0 bug — grant not idempotent
Detect:   409 on PUT /grants with duplicate key error
Fix:      GET /grants first, only PUT if not exists
Pattern:  Use safe_grant_privilege() in all ETL setup scripts
Status:   Known bug — may be fixed in future Polaris versions
